# 片上单元测试 (On-Chip Unit Test)

本 Notebook 复用 `rtl/tb/lite_bd/module_tb/golden_module_tb.py` 的数据生成逻辑，
通过 Windows XDMA 驱动 (`tests/bin/xdma_rw.exe`) 在 FPGA 板上执行 module-level 测试。

**流程概述:**
1. 生成测试数据 (inst.hex / preload.txt / checks.txt / expected.hex)
2. 通过 XDMA H2C 上传数据到 FPGA 各存储区
3. 写寄存器启动 INST_Decoder
4. 轮询完成状态
5. 通过 XDMA C2H 读回结果并与 golden 比对

## 0. 环境检查

In [ ]:
import sys
from pathlib import Path

# 确保 unit-tb 目录在路径中
UNIT_TB_DIR = Path(".").resolve()
if str(UNIT_TB_DIR) not in sys.path:
    sys.path.insert(0, str(UNIT_TB_DIR))

REPO_ROOT = UNIT_TB_DIR.parent.parent.parent
print(f"Repo root: {REPO_ROOT}")
print(f"Unit-TB dir: {UNIT_TB_DIR}")

In [ ]:
from xdma_win import XDMAWin, XDMA_RW_EXE, REGS_BASE, REG_DECODER_STATUS

# 验证 xdma_rw.exe 存在
assert XDMA_RW_EXE.exists(), f"xdma_rw.exe not found: {XDMA_RW_EXE}"
print(f"xdma_rw.exe: {XDMA_RW_EXE} ✓")

# 快速通信测试: 读 DECODER_STATUS 寄存器
xdma = XDMAWin(verbose=True)
status = xdma.read_u32(REGS_BASE + REG_DECODER_STATUS)
print(f"\nDECODER_STATUS = 0x{status:08x}")
print("XDMA 通信正常 ✓" if status != 0xDEADBEEF else "WARNING: 读到异常值")

## 1. 生成测试数据

使用 `golden_module_tb.py` 为指定 case 生成完整测试向量。

可选 case 列表:
- `dcim_matmul`: DCIM 矩阵乘 (dcim_tiny_1x1, conv3_s2_c32_to64, ...)
- `qa`: 量化单元 (qa_c16_signed, qa_c64_clip, ...)
- `dqa`: 反量化单元 (dqa_c16_small, dqa_c32_mid, ...)
- `im2col`: im2col 变换
- `conv_pipeline`: 单层 conv 全链路 (im2col→CDMA→DCIM→DQA→QA)
- `mini_network`: 多层网络 (2-3 conv + residual)

In [ ]:
from gen_data import generate_case, list_cases, list_modules

# 查看所有可用模块
print("Available modules:", list_modules())

In [ ]:
# 查看某个模块的所有可用 variant
MODULE_CASE = "dcim_matmul"  # ← 修改这里选择模块
variants = list_cases(MODULE_CASE)
print(f"\n{MODULE_CASE} variants:")
for v in variants:
    print(f"  • {v}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 配置: 选择要测试的 case
# ═══════════════════════════════════════════════════════════════════
MODULE_CASE    = "dcim_matmul"     # 模块类型
MODULE_VARIANT = "dcim_tiny_1x1"   # 具体用例 (最小规模, 适合首次验证)
QUANT          = "int8"            # 量化模式: int8 / int16
# ═══════════════════════════════════════════════════════════════════

run_dir = generate_case(MODULE_CASE, MODULE_VARIANT, quant=QUANT)
print(f"\n生成完成! 运行目录: {run_dir}")
print("\n文件列表:")
for f in sorted(run_dir.iterdir()):
    print(f"  {f.name:25s} ({f.stat().st_size:>8,} bytes)")

## 2. 检查生成的数据

In [ ]:
# 查看 preload.txt: 需要上传到 FPGA 的数据
print("=== preload.txt ===")
print((run_dir / "preload.txt").read_text())

print("\n=== checks.txt ===")
print((run_dir / "checks.txt").read_text())

print("\n=== inst.hex (前 10 行) ===")
lines = (run_dir / "inst.hex").read_text().splitlines()
print(f"总指令数: {len(lines)} words")
for l in lines[:10]:
    print(f"  {l}")

## 3. 上传数据到 FPGA

按 preload.txt 将各 hex 文件转为二进制，通过 XDMA H2C 写入对应物理地址。

In [ ]:
from xdma_win import ChipRunnerWin

runner = ChipRunnerWin(verbose=True)

# Step 1: 上传预加载数据 (weights, activations, WB 等)
runner.upload_preload(run_dir)

In [ ]:
# Step 2: 上传指令到 INST_BRAM
n_words = runner.upload_inst(run_dir)
print(f"\n指令已上传: {n_words} words")

## 4. 执行并等待完成

In [ ]:
# Step 3: 启动 INST_Decoder
runner.start_decoder(n_words)

# Step 4: 等待执行完成
runner.poll_done(timeout_s=10.0)
print("\n执行完成!")

## 5. 读回结果并比对 Golden

In [ ]:
# Step 5: 从 FPGA 读回结果, 与 expected.hex 逐 word 比对
results = runner.read_check(run_dir)

print("\n" + "="*60)
print("Results Summary")
print("="*60)
all_pass = True
for r in results:
    status = "PASS ✓" if r["pass"] else "FAIL ✗"
    print(f"  {r['name']:30s} {status}  ({r['passed']}/{r['total_words']} words)")
    if not r["pass"]:
        all_pass = False
        m = r["first_mismatch"]
        print(f"    First mismatch at word {m['word']}:")
        print(f"      expected: {m['expected']}")
        print(f"      got:      {m['got']}")

print("\n" + ("ALL PASS ✓" if all_pass else "SOME FAILED ✗"))

## 6. 一键执行 (完整流程)

以上 Step 1~5 封装为 `run_case()` 一键调用:

In [ ]:
from xdma_win import ChipRunnerWin
from gen_data import generate_case

# 一键: 生成 + 上传 + 执行 + 比对
run_dir = generate_case("dcim_matmul", "dcim_tiny_1x1", quant="int8")
runner = ChipRunnerWin(verbose=True)
results = runner.run_case(run_dir, timeout_s=10.0)

## 7. 批量测试

复用 module_tb 的 smoke suite 定义, 逐个执行并汇总:

In [ ]:
from xdma_win import ChipRunnerWin
from gen_data import generate_case

# ═══════════════════════════════════════════════════════════════════
# 批量配置
# ═══════════════════════════════════════════════════════════════════
BATCH = [
    # (module_case, variant, quant)
    ("dcim_matmul", "dcim_tiny_1x1", "int8"),
    ("dcim_matmul", "conv6_s2_c3_to16", "int8"),
    ("dcim_matmul", "conv3_s2_c32_to64", "int8"),
    ("qa", "qa_c16_signed", "int8"),
    ("dqa", "dqa_c16_small", "int8"),
    ("im2col", "im2col_6x6_s2_c3", "int8"),
]
STOP_ON_FAIL = False
# ═══════════════════════════════════════════════════════════════════

runner = ChipRunnerWin(verbose=False)
summary = []

for case, variant, quant in BATCH:
    try:
        run_dir = generate_case(case, variant, quant=quant)
        results = runner.run_case(run_dir, timeout_s=15.0)
        passed = all(r["pass"] for r in results)
        summary.append((f"{case}/{variant}", "PASS" if passed else "FAIL", results))
        if not passed and STOP_ON_FAIL:
            break
    except Exception as e:
        summary.append((f"{case}/{variant}", f"ERROR: {e}", []))
        if STOP_ON_FAIL:
            break

print("\n" + "="*70)
print("Batch Results")
print("="*70)
n_pass = 0
for name, status, _ in summary:
    icon = "✓" if status == "PASS" else "✗"
    print(f"  {icon} {name:40s} {status}")
    if status == "PASS":
        n_pass += 1

print(f"\nTotal: {n_pass}/{len(summary)} PASS")

## 8. 调试工具

当测试失败时, 可用以下工具进行诊断:

In [ ]:
import numpy as np
from xdma_win import (
    XDMAWin, hex_to_bin,
    REGS_BASE, INST_BASE, VPU_BUF_BASE, TILE_IBUF_BASE, TILE_OBUF_BASE,
    REG_STATUS, REG_DECODER_CTRL, REG_INST_COUNT, REG_DECODER_STATUS,
)

xdma = XDMAWin(verbose=True)

# 读取所有控制寄存器
print("=== VPU_AXI_Regs ===")
print(f"  STATUS         (0x04) = 0x{xdma.read_u32(REGS_BASE + 0x04):08x}")
print(f"  DECODER_CTRL   (0x38) = 0x{xdma.read_u32(REGS_BASE + 0x38):08x}")
print(f"  INST_COUNT     (0x3C) = 0x{xdma.read_u32(REGS_BASE + 0x3C):08x}")
print(f"  DECODER_STATUS (0x40) = 0x{xdma.read_u32(REGS_BASE + 0x40):08x}")

In [ ]:
# 手动读取任意 FPGA 地址 (用于 debug)
READ_ADDR = TILE_OBUF_BASE  # ← 修改地址
READ_WORDS = 4               # ← 读多少个 128-bit word

raw = xdma.read(READ_ADDR, READ_WORDS * 16)
for i in range(READ_WORDS):
    word = raw[i*16:(i+1)*16]
    print(f"  [{i:3d}] {word.hex()}")

In [ ]:
# 手动写入单个 32-bit 寄存器 (用于 debug)
# xdma.write_u32(REGS_BASE + REG_DECODER_CTRL, 0)  # 取消注释执行